In [1]:
import csv, time, json, ast
from seleniumbase import Driver
from pprint import pprint
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import shutil
import csv

In [10]:
brand = 'Atomic'
browser = Driver(uc=True, incognito=True)
browser_wait = WebDriverWait(browser, 60)
# browser.maximize_window()

In [3]:
browser.get('https://atomic-shop.eu/collections')

In [6]:
browser.get('https://atomic-shop.eu/collections')
time.sleep(3)
ptype_list = []

ptypes = browser.find_elements(by=By.CSS_SELECTOR, value='.header__bottom-navigation li.linklist__item a')

for i in ptypes:
    ptype = {}
    type_link = i.get_attribute('href').strip()
    type = i.get_attribute('innerHTML').strip()
    # print(type)
    ptype[type] = type_link
    ptype_list.append(ptype)

print(ptype_list)

[{'Brake Pads': 'https://atomic-shop.eu/collections/brake-pads'}, {'Brake Rotors': 'https://atomic-shop.eu/collections/brake-rotors'}, {'Brake Calipers': 'https://atomic-shop.eu/collections/brake-calipers'}, {'Brake Kits': 'https://atomic-shop.eu/collections/brake-kits'}, {'Brake Lines': 'https://atomic-shop.eu/collections/brake-lines'}, {'Brake Cooling': 'https://atomic-shop.eu/collections/brake-cooling'}, {'Moto Brake Pads': 'https://atomic-shop.eu/collections/moto-brake-pads'}, {'Brake Misc': 'https://atomic-shop.eu/collections/brake-misc'}, {'Coilover Kits': 'https://atomic-shop.eu/collections/coilover-kits'}, {'Lowering Springs': 'https://atomic-shop.eu/collections/lowering-springs'}, {'Shock Absorbers': 'https://atomic-shop.eu/collections/shock-absorbers'}, {'Shock Mounts': 'https://atomic-shop.eu/collections/shock-mounts'}, {'Control Arms': 'https://atomic-shop.eu/collections/control-arms'}, {'Strut Braces': 'https://atomic-shop.eu/collections/strut-braces'}, {'Sway Bars': 'http

In [14]:
products_list = []
with open(f'atomic-products.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['type','vendor', 'link'])

    for itype in ptype_list:
        for key, value in itype.items():
            type = key
            link = value

        proceed = True
        current_page = 1

        while(proceed):
            url_product_page = link + "?page=" + str(current_page)
            browser.get(url_product_page)

            products = browser.find_elements(by=By.CSS_SELECTOR, value='.product-item-meta')

            for i in products:
                product = i.find_element(by=By.CSS_SELECTOR, value='.product-item-meta__title').get_attribute('href').strip()
                vendor = i.find_element(by=By.CSS_SELECTOR, value='.product-item-meta__vendor').get_attribute('innerText').strip()

                writer.writerow([type, vendor, product])
                products_list.append(product)

            products_list = list(set(products_list))
            print(len(products_list))

            product_existence = browser.find_elements(by=By.CSS_SELECTOR, value='.product-item-meta')
            if product_existence == []:
                proceed = False
            else:
                current_page += 1

32
64
96
128
160
192
224
256
288
320
352
384
416
448
480
512
544
576
608
640
672
704
736
768
800
832
864
896
928
960
992
1024
1056
1088
1120
1152
1184
1216
1248
1280
1280
1312
1344
1376
1408
1440
1472
1504
1536
1568
1600
1632
1664
1696
1728
1760
1792
1824
1856
1888
1920
1952
1984
2016
2048
2080
2112
2144
2176
2208
2240
2272
2304
2336
2368
2400
2432
2464
2496
2528
2560
2592
2624
2656
2688
2720
2752
2784
2816
2848
2880
2912
2944
2976
3008
3040
3072
3104
3136
3168
3200
3232
3264
3296
3328
3328
3360
3392
3424
3456
3488
3520
3552
3584
3616
3648
3680
3712
3744
3776
3808
3840
3872
3904
3936
3968
4000
4032
4064
4096
4128
4160
4192
4224
4256
4288
4320
4352
4384
4416
4448
4480
4512
4544
4576
4608
4640
4672
4704
4736
4768
4800
4832
4864
4896
4928
4960
4992
5024
5056
5088
5120
5152
5184
5216
5248
5280
5312
5344
5376
5408
5440
5472
5504
5536
5568
5600
5632
5664
5696
5728
5760
5792
5824
5856
5888
5920
5952
5984
6016
6048
6080
6112
6144
6176
6208
6240
6272
6304
6336
6368
6400
6432
6464
6496
6528
6560

In [7]:
header = ['link', 'vendor', 'product type', 'title', 'sku', 'price', 'images', 'desc', 'compatibility']
with open(f'atomic-scrape.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(header)

    df1 = pd.read_csv(r'atomic-products.csv', dtype=str)
    for index, row in df1.iterrows():
        link = row['link']
        vendor = row['vendor']
        product_type = row['type']

        browser.get(link)
        try:
            title = browser.find_element(by=By.CSS_SELECTOR, value='.product-meta__title').get_attribute('innerText').strip()
            sku = browser.find_element(by=By.CSS_SELECTOR, value='.product-meta__sku-number').get_attribute('innerText').strip()
            price = browser.find_element(by=By.CSS_SELECTOR, value='.price--large').get_attribute('innerHTML').replace('<span class="visually-hidden">Sale price</span>','').strip()
            images = browser.find_elements(by=By.CSS_SELECTOR, value='.product__media-image-wrapper img')
            image_list = []
            for i in images:
                image = i.get_attribute('src')
                image_list.append(image)

            try:
                desc = browser.find_element(by=By.CSS_SELECTOR, value='.product-tabs__tab-item-content').get_attribute('innerHTML')
            except:
                desc = ''
                
            try:
                compatibility = browser.find_element(by=By.CSS_SELECTOR, value='.metafield-rich_text_field').get_attribute('innerHTML')
            except:
                compatibility = ''

            if compatibility == desc:
                desc = ''
            
            writer.writerow([link, vendor, product_type, title, sku, price, image_list, desc, compatibility])
            print([link, vendor, product_type, title, sku, price, image_list, desc, compatibility])
        except Exception as e:
            # print(f"{cell}: {e}")
            print(f"{link}: {str(e).splitlines()[0]}")


['https://atomic-shop.eu/products/bell-2181025-interior-kit-for-karting-helmet-kc7-cmr-carbon-cmr2016-blue-58', 'BELL', 'Karting Helmets', 'BELL 2181025 INTERIOR KIT FOR KARTING HELMET KC7-CMR CARBON CMR2016 - BLUE 58', '2181025', '€590,00', ['https://atomic-shop.eu/cdn/shop/files/image-coming-soon_870d770f-ef3c-40ec-80da-759fba2d3e73.png?v=1735692632&width=1800'], '', '']
['https://atomic-shop.eu/products/bell-2181032-interior-kit-for-karting-helmet-kc7-cmr-carbon-cmr2016-red-55', 'BELL', 'Karting Helmets', 'BELL 2181032 INTERIOR KIT FOR KARTING HELMET KC7-CMR CARBON CMR2016 - RED 55', '2181032', '€590,00', ['https://atomic-shop.eu/cdn/shop/files/image-coming-soon_b6b2254c-5498-4d30-9156-3e68c53bbc33.png?v=1735692650&width=1800'], '', '']
['https://atomic-shop.eu/products/bell-2181016-interior-kit-for-karting-helmet-kc7-cmr-carbon-cmr2016-black-59', 'BELL', 'Karting Helmets', 'BELL 2181016 INTERIOR KIT FOR KARTING HELMET KC7-CMR CARBON CMR2016 - BLACK 59', '2181016', '€405,00', ['http

HTML

In [6]:
csv_path = r"atomic-scrape_pagid.csv"
scrape_df = pd.read_csv(csv_path, encoding='ISO-8859-1').fillna('')

final_data = []

for index0, row in scrape_df.iterrows():

    title = row['title']
    sku = f"'{row['sku']}"
    ven = row['vendor']
    vendor = ven.title()
    product_type = row['product type']
    price = row['price']
    images = ast.literal_eval(row['images'])
    desc = row['desc']
    compatibility = row['compatibility']

    if desc == '':
        desc = f'This is {title}'

    if compatibility == '':
        if 'FOR ' in title:
            # print('yes')
            comp = title.split('FOR ')[-1]
            # print(comp)
            compatibility = f'<ul><li>{comp}</li></ul>'
            # print(compatibility)
        else: 
            # print('No')
            compatibility = ''

    handle = (re.sub(r'[^a-zA-Z0-9\n\.]', '-', title).replace(".", "-").replace("---", "-").replace("--", "-")).lower()
    if handle.endswith('-'):
        handle = handle[:-1]

    def description(desc, compatibility, sku, vendor):
        return f"""<h4><strong>Description</strong></h4>{desc}
        <p>&nbsp;</p>
        <h4>Compatibility</h4>{compatibility}<p>Feel free to contact us at info@mlperformance.co.uk should you wish to double check!</p>
        <p>&nbsp;</p>
        <h4>Compatibility Check</h4><p>To ensure the part(s) you have ordered fits your vehicle, we run a compatibility check prior to dispatch. We can do this either using your registration number (UK) or the last 7 digits of your VIN. Simply enter your car details prior to checkout.</p>
        <p>&nbsp;</p>
        <h4>Part Number</h4><p>{ven}-{sku.replace("'","")}</p>
        <p>&nbsp;</p>
        <h4>More Information</h4><p><strong>Manufactured by</strong></p><p>{vendor}</p>"""

    # Loop through images
    for index, image in enumerate(images, start=0):

        info = {}
        
        info['Handle'] = handle
        info['Title'] = title
        info['Body (HTML)'] = description(desc, compatibility, sku, vendor).replace("\n", "")
        info['Vendor'] = vendor
        info['Standardized Product Type'] = product_type
        info['Custom Product Type'] = None
        info['Tags'] = f"Uploaded by_Muazzim, Brand_{vendor}, Product Type_"
        info['Published'] = "TRUE"
        info['Option1 Name'] = ''
        info['Option1 Value'] = ''
        info['Option2 Name'] = ''
        info['Option2 Value'] = ''
        info['Option3 Name'] = ''
        info['Option3 Value'] = ''

        info['Variant SKU'] = f"""{ven}-{sku.replace("'","")}"""
        info['Variant Grams'] = None
        info['Variant Inventory Tracker'] = "shopify"
        info['Variant Inventory Policy'] = 'continue'
        info['Variant Fulfillment Service'] = 'manual'
        info['Variant Price'] = None
        info['Variant Compare At Price'] = price
        info['Variant Requires Shipping'] = 'TRUE'
        info['Variant Taxable'] = 'TRUE'
        info['Variant Barcode'] = sku.replace("''","'")

        # For each image, create a new entry
        info['Image Src'] = image
        info['Image Position'] = index + 1
        info['Image Alt Text'] = title
        info['Gift Card'] = None
        info['SEO Title'] = title
        info['SEO Description'] = 'Get ' + title + ' for your car to get your desired looks and performance from ML Performance at the lowest price with FREE UK shipping & next day delivery on in stock items. Very cheap prices & good service.'
        info['Google Shopping / Google Product Category'] = None
        info['Google Shopping / Gender'] = None
        info['Google Shopping / Age Group'] = None
        info['Google Shopping / MPN'] = sku.replace("''","'")
        info['Google Shopping / AdWords Grouping'] = None
        info['Google Shopping / AdWords Labels'] = None
        info['Google Shopping / Condition'] = 'new'
        info['Google Shopping / Custom Product'] = None
        info['Google Shopping / Custom Label 0'] = None
        info['Google Shopping / Custom Label 1'] = None
        info['Google Shopping / Custom Label 2'] = None
        info['Google Shopping / Custom Label 3'] = None
        info['Google Shopping / Custom Label 4'] = None
        info['Variant Image'] = None
        info['Variant Weight Unit'] = 'kg'
        info['Variant Tax Code'] = 8708949900
        info['Cost per item'] = None
        info['Margins'] = None
        info['Price / International'] = None
        info['Compare At Price / International'] = None
        info['Status'] = 'active'

        # Append the new dictionary to the final_data list
        final_data.append(info)

final_df = pd.DataFrame(final_data)

output_path = os.path.join("Atomic_pagid-HTML.csv")
final_df.to_csv(output_path, index=False)

print('File saved and moved to desired folder')

File saved and moved to desired folder


Price

In [4]:
browser.get('https://atomic-shop.eu/collections?country=fr')

In [12]:
header = ['link', 'sku', 'price']
with open(f'atomic-sku.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(header)

    df1 = pd.read_csv(r'atomic-products3.csv', dtype=str)
    for index, row in df1.iterrows():
        link = row['link']

        browser.get(f'{link}?country=fr')
        time.sleep(1)
        try:
            sku = browser.find_element(by=By.CSS_SELECTOR, value='.product-meta__sku-number').get_attribute('innerText').strip()
            price = browser.find_element(by=By.CSS_SELECTOR, value='.price--large').get_attribute('innerHTML').replace('<span class="visually-hidden">Sale price</span>','').strip()
            
            writer.writerow([link, f"'{sku}", price])
            print([link, sku, price])

        except Exception as e:
            print(f"{link}: {str(e).splitlines()[0]}")


['https://atomic-shop.eu/products/pfc-025-0001-brake-fluid-z-rated-16oz-steel-can', '025.0001', '€15,56']
['https://atomic-shop.eu/products/pfc-025-0037-racing-brake-fluid-rh665-500ml', '025.0037', '€39,48']
['https://atomic-shop.eu/products/pfc-025-0038-brake-fluid-12-x-500ml-in-case', '025.0038', '€434,39']
['https://atomic-shop.eu/products/pfc-025-0053-caliper-abutment-kit-a-786-kit-pf', '025.0053', '€8,39']
['https://atomic-shop.eu/products/pfc-025-0169-brake-fluid-5-liter-bottle', '025.0169', '€28,60']
['https://atomic-shop.eu/products/pfc-025-0170-brake-fluid-12-x-5-liter-in-case', '025.0170', '€342,94']
['https://atomic-shop.eu/products/pfc-026-0001-professional-twin-port-brake-bleeder-bottle-pfc', '026.0001', '€140,68']
['https://atomic-shop.eu/products/pfc-032-0003-temp-smart-temp-signaling-paint', '032.0003', '€151,07']
['https://atomic-shop.eu/products/pfc-032-0007-temperature-caliper-labels-121c-to-280c-book-of-10-labels', '032.0007', '€64,98']
['https://atomic-shop.eu/prod

In [2]:
import csv
import pandas as pd
import re
import os

csv_path = r"atomic-a.csv"
scrape_df = pd.read_csv(csv_path, encoding='ISO-8859-1').fillna('')

output_path = os.path.join("A-HTML.csv")

# Open the output CSV file for writing
with open(output_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=[
        'Handle', 'Title', 'Body (HTML)', 'Vendor', 'Standardized Product Type', 
        'Custom Product Type', 'Tags', 'Published', 'Option1 Name', 'Option1 Value', 
        'Option2 Name', 'Option2 Value', 'Option3 Name', 'Option3 Value', 'Variant SKU', 
        'Variant Grams', 'Variant Inventory Tracker', 'Variant Inventory Policy', 
        'Variant Fulfillment Service', 'Variant Price', 'Variant Compare At Price', 
        'Variant Requires Shipping', 'Variant Taxable', 'Variant Barcode', 'Image Src', 
        'Image Position', 'Image Alt Text', 'Gift Card', 'SEO Title', 'SEO Description', 
        'Google Shopping / Google Product Category', 'Google Shopping / Gender', 
        'Google Shopping / Age Group', 'Google Shopping / MPN', 
        'Google Shopping / AdWords Grouping', 'Google Shopping / AdWords Labels', 
        'Google Shopping / Condition', 'Google Shopping / Custom Product', 
        'Google Shopping / Custom Label 0', 'Google Shopping / Custom Label 1', 
        'Google Shopping / Custom Label 2', 'Google Shopping / Custom Label 3', 
        'Google Shopping / Custom Label 4', 'Variant Image', 'Variant Weight Unit', 
        'Variant Tax Code', 'Cost per item', 'Margins', 'Price / International', 
        'Compare At Price / International', 'Status'
    ])

    # Write the header to the CSV
    writer.writeheader()

    for index0, row in scrape_df.iterrows():
        title = row['title']
        sku = row['sku']
        vendor = row['vendor']
        product_type = row['product type']
        price = row['price']
        images = row['images']
        desc = row['desc']
        compatibility = row['compatibility']

        if desc == '':
            desc = f'This is {title}'

        if compatibility == '':
            if 'FOR ' in title:
                comp = title.split('FOR ')[-1]
                compatibility = f'<ul><li>{comp}</li></ul>'
            else:
                compatibility = ''

        handle = (re.sub(r'[^a-zA-Z0-9\n\.]', '-', title).replace(".", "-").replace("---", "-").replace("--", "-")).lower()
        if handle.endswith('-'):
            handle = handle[:-1]

        def description(desc, compatibility, sku, vendor):
            return f"""<h4><strong>Description</strong></h4>{desc}
            <p>&nbsp;</p>
            <h4>Compatibility</h4>{compatibility}<p>Feel free to contact us at info@mlperformance.co.uk should you wish to double check!</p>
            <p>&nbsp;</p>
            <h4>Compatibility Check</h4><p>To ensure the part(s) you have ordered fits your vehicle, we run a compatibility check prior to dispatch. We can do this either using your registration number (UK) or the last 7 digits of your VIN. Simply enter your car details prior to checkout.</p>
            <p>&nbsp;</p>
            <h4>Part Number</h4><p>{sku}</p>
            <p>&nbsp;</p>
            <h4>More Information</h4><p><strong>Manufactured by</strong></p><p>{vendor}</p>"""

        # Loop through images
        for index, image in enumerate(images, start=0):
            info = {
                'Handle': handle,
                'Title': title,
                'Body (HTML)': description(desc, compatibility, sku, vendor).replace("\n", ""),
                'Vendor': vendor,
                'Standardized Product Type': product_type,
                'Custom Product Type': None,
                'Tags': f"Uploaded by_Muazzim, Brand_{vendor}, Product Type_",
                'Published': "TRUE",
                'Option1 Name': '',
                'Option1 Value': '',
                'Option2 Name': '',
                'Option2 Value': '',
                'Option3 Name': '',
                'Option3 Value': '',
                'Variant SKU': sku,
                'Variant Grams': None,
                'Variant Inventory Tracker': "shopify",
                'Variant Inventory Policy': 'continue',
                'Variant Fulfillment Service': 'manual',
                'Variant Price': None,
                'Variant Compare At Price': price,
                'Variant Requires Shipping': 'TRUE',
                'Variant Taxable': 'TRUE',
                'Variant Barcode': sku,
                'Image Src': image,
                'Image Position': index + 1,
                'Image Alt Text': title,
                'Gift Card': None,
                'SEO Title': title,
                'SEO Description': 'Get ' + title + ' for your car to get your desired looks and performance from ML Performance at the lowest price with FREE UK shipping & next day delivery on in stock items. Very cheap prices & good service.',
                'Google Shopping / Google Product Category': None,
                'Google Shopping / Gender': None,
                'Google Shopping / Age Group': None,
                'Google Shopping / MPN': sku,
                'Google Shopping / AdWords Grouping': None,
                'Google Shopping / AdWords Labels': None,
                'Google Shopping / Condition': 'new',
                'Google Shopping / Custom Product': None,
                'Google Shopping / Custom Label 0': None,
                'Google Shopping / Custom Label 1': None,
                'Google Shopping / Custom Label 2': None,
                'Google Shopping / Custom Label 3': None,
                'Google Shopping / Custom Label 4': None,
                'Variant Image': None,
                'Variant Weight Unit': 'kg',
                'Variant Tax Code': 8708949900,
                'Cost per item': None,
                'Margins': None,
                'Price / International': None,
                'Compare At Price / International': None,
                'Status': 'active'
            }
            writer.writerow(info)

print('File saved and moved to desired folder')


KeyboardInterrupt: 

['PFC BRAKES', 'AP RACING', 'UNPLUGGED PERFORMANCE', 'DBA', 'ARD', 'AMS', 'PERRIN', 'TILTON', 'RADIUM', 'COBB', 'WIECHERS', 'SACHS PERFORMANCE', 'A.I.TECH', 'DESIGN ENGINEERING (DEI)', 'COMETIC', 'TUBI STYLE', 'MANLEY', 'FERREA', 'JE PISTONS', 'CP', 'K1', 'DODSON', 'DEATSCHWERKS', 'WALBRO', 'FUELAB', 'QUAIFE', 'SCHROTH', 'ECUTEK', 'HP TUNERS', 'LINK ECU', 'CARGRAPHIC', 'SYVECS', 'INNOVATE', 'RACELOGIC', 'RN VISION', 'MURRAY']